In [1]:
import findspark

In [2]:
findspark.init()

In [3]:
from pyspark.sql import SparkSession

In [4]:
spark = SparkSession.builder.appName("first code").master('local[3]').config('spark.driver.host','127.0.0.1')\
.config('spark.ui.port',0).getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/29 02:08:41 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [5]:
spark

In [6]:
type(spark.sparkContext)

pyspark.context.SparkContext

In [7]:
ppl= spark.read.format('json').load(r'/home/AzureAdmin/data/people.json')   #eager execution    

In [8]:
type(ppl)

pyspark.sql.dataframe.DataFrame

In [9]:
ppl.printSchema()

root
 |-- age: long (nullable = true)
 |-- name: string (nullable = true)
 |-- pcode: string (nullable = true)



In [10]:
ppl.columns

['age', 'name', 'pcode']

In [11]:
ppl.schema

StructType([StructField('age', LongType(), True), StructField('name', StringType(), True), StructField('pcode', StringType(), True)])

In [12]:
ppl.show()

+----+-------+-----+
| age|   name|pcode|
+----+-------+-----+
|NULL|  Alice|94304|
|  30|Brayden|94304|
|  19|  Carla|10036|
|  46|  Diana| NULL|
|NULL|Etienne|94104|
+----+-------+-----+



In [13]:
acc_par = spark.read.format('parquet').load(r'/home/AzureAdmin/data/accounts_parquet/accounts_parquet')


In [14]:
acc_par.printSchema()

root
 |-- acct_num: integer (nullable = true)
 |-- acct_create_dt: long (nullable = true)
 |-- acct_close_dt: long (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- address: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- zipcode: string (nullable = true)
 |-- phone_number: string (nullable = true)
 |-- created: long (nullable = true)
 |-- modified: long (nullable = true)



In [15]:
acc_par.show()

+--------+--------------+-------------+----------+---------+--------------------+-------------+-----+-------+------------+-------------+-------------+
|acct_num|acct_create_dt|acct_close_dt|first_name|last_name|             address|         city|state|zipcode|phone_number|      created|     modified|
+--------+--------------+-------------+----------+---------+--------------------+-------------+-----+-------+------------+-------------+-------------+
|   32441| 1341603329000|         NULL|     Peter|  Zachary|  1891 Jessie Street|   Long Beach|   CA|  90743|  5628719002|1395174646000|1395174646000|
|   32442| 1345273687000|         NULL|     Duane|     Ruiz| 1023 Simpson Street|     Pasadena|   CA|  91051|  6262472463|1395174646000|1395174646000|
|   32443| 1347189332000|         NULL|   Desiree|    Beall|    4830 Davis Place|         Reno|   NV|  89468|  7758006922|1395174646000|1395174646000|
|   32444| 1353649861000|         NULL|     Molly|   Pinder|    708 Locust Court|    Las Vegas

In [16]:
acc_par.count()

97320

In [17]:
ppl.describe().show()

+-------+------------------+-------+------------------+
|summary|               age|   name|             pcode|
+-------+------------------+-------+------------------+
|  count|                 3|      5|                 4|
|   mean|31.666666666666668|   NULL|           73187.0|
| stddev|13.576941236277534|   NULL|42100.772233614276|
|    min|                19|  Alice|             10036|
|    max|                46|Etienne|             94304|
+-------+------------------+-------+------------------+



In [18]:
latlon = spark.read.format('csv').option('sep','\t')\
.option('inferSchema',True)\
.load(r'/home/AzureAdmin/data/latlon.tsv')

In [19]:
latlon.printSchema()

root
 |-- _c0: integer (nullable = true)
 |-- _c1: double (nullable = true)
 |-- _c2: double (nullable = true)



In [20]:
latlon.show(3)

+---+---------+----------+
|_c0|      _c1|       _c2|
+---+---------+----------+
|210|43.005895|-71.013202|
|211|43.005895|-71.013202|
|212|43.005895|-71.013202|
+---+---------+----------+
only showing top 3 rows



In [21]:
from pyspark.sql.types import *

In [22]:
sch = StructType([StructField("citycode",IntegerType(),True),\
                 StructField("lattitude",FloatType(),True),\
                 StructField("longitude",FloatType(),True)])

In [23]:
latlon = spark.read.format('csv').option('sep','\t')\      #lazy execution
.schema(sch).load(r'/home/AzureAdmin/data/latlon.tsv')

SyntaxError: unexpected character after line continuation character (2603299636.py, line 1)

In [ ]:
latlon.printSchema()


In [ ]:
latlon.show(4)

In [ ]:
res = ppl.select ("name","age","pcode")
res.show()

In [ ]:
res = ppl.select(ppl["name"],ppl["age"],ppl["pcode"])
res.show()

In [ ]:
res = ppl.select (ppl.name,ppl.age,ppl.pcode)
res.show()

In [ ]:
from pyspark.sql.functions import col

In [ ]:
res = ppl.select(col("name"),col("age"),col("pcode"))
res.show()

In [ ]:
res = ppl.where(col("age")>30)
res.show()

In [ ]:
acc_par.where(col("state") == 'CA').select(col("first_name"),col("state")).show()

In [ ]:
ppl.orderBy("age").show()

In [ ]:
ppl.orderBy(col("age").desc_nulls_last()).show()

In [ ]:
ppl.orderBy("pcode",col("age").desc_nulls_last()).show()

In [26]:
gp = acc_par.groupby('state').count()

In [27]:
gp.show()

+-----+-----+
|state|count|
+-----+-----+
|   AZ| 8614|
|   OR|10166|
|   NV| 8093|
|   CA|70447|
+-----+-----+



In [28]:
acc_par.groupby('state','city').count().show()

+-----+---------------+-----+
|state|           city|count|
+-----+---------------+-----+
|   CA|        Oakland| 2718|
|   CA|       Stockton| 2536|
|   AZ|      Flagstaff|  804|
|   CA|     Long Beach| 3380|
|   NV|      Las Vegas| 3066|
|   AZ|        Phoenix| 3142|
|   OR|         Eugene| 1162|
|   CA|       Torrance| 1071|
|   CA|North Hollywood| 1126|
|   CA|       San Jose| 2550|
|   CA|      Palo Alto| 1372|
|   CA|       Richmond| 1361|
|   AZ|        Kingman|  795|
|   CA|       Alhambra| 1106|
|   NV|            Ely|  987|
|   CA|         Fresno| 3342|
|   CA|       Berkeley| 1310|
|   CA|     Marysville| 1292|
|   NV|           Elko| 1001|
|   CA|         Oxnard| 1145|
+-----+---------------+-----+
only showing top 20 rows



In [30]:
from pyspark.sql.functions import col
acc_par.groupby('state','city').count().where(col('count')>5000).show()

+-----+----------+-----+
|state|      city|count|
+-----+----------+-----+
|   CA|Sacramento| 5318|
+-----+----------+-----+



In [31]:
ppl.show()

+----+-------+-----+
| age|   name|pcode|
+----+-------+-----+
|NULL|  Alice|94304|
|  30|Brayden|94304|
|  19|  Carla|10036|
|  46|  Diana| NULL|
|NULL|Etienne|94104|
+----+-------+-----+



In [34]:
ppl.orderBy('name').show()

+----+-------+-----+
| age|   name|pcode|
+----+-------+-----+
|NULL|  Alice|94304|
|  30|Brayden|94304|
|  19|  Carla|10036|
|  46|  Diana| NULL|
|NULL|Etienne|94104|
+----+-------+-----+



In [35]:
ppl.orderBy(ppl['name'].desc()).show()

+----+-------+-----+
| age|   name|pcode|
+----+-------+-----+
|NULL|Etienne|94104|
|  46|  Diana| NULL|
|  19|  Carla|10036|
|  30|Brayden|94304|
|NULL|  Alice|94304|
+----+-------+-----+



In [41]:
res=ppl.dropna(subset=['pcode'])
res.show()

+----+-------+-----+
| age|   name|pcode|
+----+-------+-----+
|NULL|  Alice|94304|
|  30|Brayden|94304|
|  19|  Carla|10036|
|NULL|Etienne|94104|
+----+-------+-----+



In [42]:
ppl.na.drop(how='all').show()

+----+-------+-----+
| age|   name|pcode|
+----+-------+-----+
|NULL|  Alice|94304|
|  30|Brayden|94304|
|  19|  Carla|10036|
|  46|  Diana| NULL|
|NULL|Etienne|94104|
+----+-------+-----+



In [43]:
ppl.na.fill(0).show()

+---+-------+-----+
|age|   name|pcode|
+---+-------+-----+
|  0|  Alice|94304|
| 30|Brayden|94304|
| 19|  Carla|10036|
| 46|  Diana| NULL|
|  0|Etienne|94104|
+---+-------+-----+



In [44]:
ppl.printSchema()

root
 |-- age: long (nullable = true)
 |-- name: string (nullable = true)
 |-- pcode: string (nullable = true)



In [45]:
ppl.printSchema()

root
 |-- age: long (nullable = true)
 |-- name: string (nullable = true)
 |-- pcode: string (nullable = true)



In [46]:
acc_par.printSchema()

root
 |-- acct_num: integer (nullable = true)
 |-- acct_create_dt: long (nullable = true)
 |-- acct_close_dt: long (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- address: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- zipcode: string (nullable = true)
 |-- phone_number: string (nullable = true)
 |-- created: long (nullable = true)
 |-- modified: long (nullable = true)



In [47]:
#N
ppl.count()

5

In [48]:
#M
acc_par.count()

97320

In [49]:
cj = ppl.crossJoin(acc_par)
cj.count()

486600

In [50]:
cj.show(4)

+----+-------+-----+--------+--------------+-------------+----------+---------+------------------+----------+-----+-------+------------+-------------+-------------+
| age|   name|pcode|acct_num|acct_create_dt|acct_close_dt|first_name|last_name|           address|      city|state|zipcode|phone_number|      created|     modified|
+----+-------+-----+--------+--------------+-------------+----------+---------+------------------+----------+-----+-------+------------+-------------+-------------+
|NULL|  Alice|94304|   32441| 1341603329000|         NULL|     Peter|  Zachary|1891 Jessie Street|Long Beach|   CA|  90743|  5628719002|1395174646000|1395174646000|
|  30|Brayden|94304|   32441| 1341603329000|         NULL|     Peter|  Zachary|1891 Jessie Street|Long Beach|   CA|  90743|  5628719002|1395174646000|1395174646000|
|  19|  Carla|10036|   32441| 1341603329000|         NULL|     Peter|  Zachary|1891 Jessie Street|Long Beach|   CA|  90743|  5628719002|1395174646000|1395174646000|
|  46|  Di

In [51]:
innerj = ppl.join(acc_par,ppl['pcode']==acc_par['zipcode'],'inner')
innerj.count()

33

In [52]:
louterj = ppl.join(acc_par,ppl['pcode']==acc_par['zipcode'],'left')
louterj.count()

35

In [53]:
Routerj = ppl.join(acc_par,ppl['pcode']==acc_par['zipcode'],'right_outer')
Routerj.count()

97331

In [54]:
fouterj = ppl.join(acc_par,ppl['pcode']==acc_par['zipcode'],'fullouter')
fouterj.count()

97333

In [55]:
lsemi = ppl.join(acc_par,ppl['pcode']==acc_par['zipcode'],'semi')
lsemi.count()

3

In [56]:
lsemi.show()

+----+-------+-----+
| age|   name|pcode|
+----+-------+-----+
|NULL|  Alice|94304|
|  30|Brayden|94304|
|NULL|Etienne|94104|
+----+-------+-----+



In [57]:
ajoin = ppl.join(acc_par,ppl['pcode']==acc_par['zipcode'],'anti')
ajoin.show()

+---+-----+-----+
|age| name|pcode|
+---+-----+-----+
| 19|Carla|10036|
| 46|Diana| NULL|
+---+-----+-----+



In [58]:
ppl.show()

+----+-------+-----+
| age|   name|pcode|
+----+-------+-----+
|NULL|  Alice|94304|
|  30|Brayden|94304|
|  19|  Carla|10036|
|  46|  Diana| NULL|
|NULL|Etienne|94104|
+----+-------+-----+



In [59]:
ppl.createOrReplaceTempView('people')
acc_par.createOrReplaceTempView('account')

In [60]:
res = spark.sql('select * from people p1 left semi join account a1 on p1.pcode= a1.zipcode')
res.show()

+----+-------+-----+
| age|   name|pcode|
+----+-------+-----+
|NULL|  Alice|94304|
|  30|Brayden|94304|
|NULL|Etienne|94104|
+----+-------+-----+



In [62]:
res = spark.sql('select count(*) as cnt from people p1 join account a1 on p1.pcode= a1.zipcode')
res.show()

+---+
|cnt|
+---+
| 33|
+---+



In [67]:
cj.write.format('json').mode('overwrite').save(r'/home/AzureAdmin/data/cjoutput')

In [68]:
ppl.show()

+----+-------+-----+
| age|   name|pcode|
+----+-------+-----+
|NULL|  Alice|94304|
|  30|Brayden|94304|
|  19|  Carla|10036|
|  46|  Diana| NULL|
|NULL|Etienne|94104|
+----+-------+-----+



In [70]:
ppl.withColumn('id',col('age')+10).show()

+----+-------+-----+----+
| age|   name|pcode|  id|
+----+-------+-----+----+
|NULL|  Alice|94304|NULL|
|  30|Brayden|94304|  40|
|  19|  Carla|10036|  29|
|  46|  Diana| NULL|  56|
|NULL|Etienne|94104|NULL|
+----+-------+-----+----+

